In [1]:

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_openai import  OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
print('importing done')

C:\Users\babab\AppData\Local\Temp\ipykernel_376\3501931971.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


importing done


In [2]:
#1 Load PDF
from langchain_community.document_loaders import PyMuPDFLoader
loader= PyMuPDFLoader('Positional versus Symbolic Attention Heads.pdf')
documents=loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\nSample from page 1:\n{documents[0].page_content[:500]}")

Total pages loaded: 30

Sample from page 1:
Positional versus Symbolic Attention Heads: Learning
Dynamics, RoPE Geometry, and Length
Generalization
Felipe Urrutia
CENIA & Faculty of Mathematics UC
Santiago, Chile
felipe.urrutia@uc.cl
Juan José Alegría
CENIA
Santiago, Chile
jotaj.8a@gmail.com
Cinthia Sanchez Macias
CENIA
Santiago, Chile
mabel.sanchez.macias@gmail.com
Jorge Salas
CENIA
Santiago, Chile
jorge.salas@cenia.cl
Cristian B. Calderon
CENIA
Santiago, Chile
cristian.buc@cenia.cl
Cristobal Rojas
IMC UC & CENIA
Santiago, Chile
luis.roj


In [3]:
#2 split the pdf
splitter= RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100   )
chunks=splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[5].page_content}")

Total chunks: 226

Sample chunk:
validate the resulting predictions in both controlled and real-world models, showing
that symbolic mechanisms extrapolate more reliably to longer sequences while
positional mechanisms face sharper limitations.
1
Introduction
Transformer-based LLMs are shaping distinct aspects of society, ranging from education [Chu et al.,
2025] to work productivity [Cambon et al., 2023]. As such, it becomes pressing to develop a profound


In [4]:
#3 load openai key in .env file and .gitignore to protect file from getting exposed outside
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
#4 Embedding
embeddings=OpenAIEmbeddings(
    api_key=os.getenv('OPENAI_API_KEY')
    )

vectorstore=Chroma.from_documents(
    documents= chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print(f"Total vectors stored: {vectorstore._collection.count()}")

Total vectors stored: 226


In [6]:
#452 is double our chunk count of 226. That is a problem.
#What happened:
#You likely ran the cell twice. Chroma appended 226 chunks on top of the existing 226 instead of overwriting. 
# Chroma's from_documents does not check for duplicates — it just adds.

## i reran it after this 452 chunks so error u are unable to see

In [7]:
#4 — Basic retriever to test before we build anything complex:
retriever= vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

In [8]:
query = "What is the difference between positional and symbolic attention heads?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
how more complex tasks are solved through their combination.
2.2
Positional and symbolic attention head scores
To assess mechanistic differences between the tasks, we use the scores introduced by Urrutia et al.
[2025], which characterize whether an attention head behaves positionally or symbolically. Informally,
a head is positional if its attention distribution is invariant to input-token permutations, and symbolic

--- Chunk 2 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbolic head behavior
can be potentially relevant at the practical scale. Indeed, empirical evidence has shown that symbolic
mechanisms are likely to have better generalization properties than positional ones [Barbero et al.,

--- Chunk 3 ---
of attention head behavior. In particular, this method reveals whet

In [ ]:
# observation = information is cut. and some chunks do talk about the query but not fully

# we will improve retriever now

In [9]:
mmr_retriever=vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k':3,
        'fetch_k':10,
        'lambda_mult':0.7
    }
)

In [10]:
query = "What is the difference between positional and symbolic attention heads?"

results = mmr_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)


--- Chunk 1 ---
how more complex tasks are solved through their combination.
2.2
Positional and symbolic attention head scores
To assess mechanistic differences between the tasks, we use the scores introduced by Urrutia et al.
[2025], which characterize whether an attention head behaves positionally or symbolically. Informally,
a head is positional if its attention distribution is invariant to input-token permutations, and symbolic

--- Chunk 2 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbolic head behavior
can be potentially relevant at the practical scale. Indeed, empirical evidence has shown that symbolic
mechanisms are likely to have better generalization properties than positional ones [Barbero et al.,

--- Chunk 3 ---
of attention head behavior. In particular, this method reveals whet

In [11]:
'''Why MMR showed no meaningful difference here:
Your 226 chunks are from a single focused research paper on one narrow topic — positional vs symbolic attention heads. 
Every chunk is already semantically close to every other chunk. MMR diversifies within the candidate pool, 
but if all 10 candidates in fetch_k are already talking about the same narrow topic, diversification has nowhere to go.
MMR shows dramatic improvement when your vector store has chunks from multiple diverse documents — different topics, different sources. 
On a single focused paper it is nearly useless.
This is an important lesson: Retriever choice depends on your data characteristics, not just your query. 
MMR is the wrong tool for a single-topic corpus.
'''

'Why MMR showed no meaningful difference here:\nYour 226 chunks are from a single focused research paper on one narrow topic — positional vs symbolic attention heads. \nEvery chunk is already semantically close to every other chunk. MMR diversifies within the candidate pool, \nbut if all 10 candidates in fetch_k are already talking about the same narrow topic, diversification has nowhere to go.\nMMR shows dramatic improvement when your vector store has chunks from multiple diverse documents — different topics, different sources. \nOn a single focused paper it is nearly useless.\nThis is an important lesson: Retriever choice depends on your data characteristics, not just your query. \nMMR is the wrong tool for a single-topic corpus.\n'

In [12]:
'''Move to parent document retriever — this will show a visible difference.
Before we write the code, this retriever needs two things we have not set up:

A docstore to store parent chunks — remember we discussed this yesterday
A different splitting strategy — parent splitter with larger chunks, child splitter with smaller chunks'''

'Move to parent document retriever — this will show a visible difference.\nBefore we write the code, this retriever needs two things we have not set up:\n\nA docstore to store parent chunks — remember we discussed this yesterday\nA different splitting strategy — parent splitter with larger chunks, child splitter with smaller chunks'

In [13]:
'''200 for child — small enough for precise retrieval on dense academic content.
1000 for parent — large enough to capture a complete idea with surrounding context.
That is roughly 5x ratio which is actually better than my suggested 3-4x for this paper given how dense the concepts are.'''

'200 for child — small enough for precise retrieval on dense academic content.\n1000 for parent — large enough to capture a complete idea with surrounding context.\nThat is roughly 5x ratio which is actually better than my suggested 3-4x for this paper given how dense the concepts are.'

In [16]:
# ParentDocument Retriever
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
# same recurvsive character splitter



In [17]:
# Two splitters
parent_splitter=RecursiveCharacterTextSplitter(chunk_size=1000)
child_splitter=RecursiveCharacterTextSplitter(chunk_size=200)

#docstore = stores parent_splitters
docstore=InMemoryStore()

# Fresh vectorstore for child embeddings
child_vectorestore=Chroma(
    collection_name='child_chunks',
    embedding_function=embeddings
)

parent_retriever=ParentDocumentRetriever(
    vectorstore=child_vectorestore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_retriever.add_documents(documents)

C:\Users\babab\AppData\Local\Temp\ipykernel_376\1022851060.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  child_vectorestore=Chroma(


In [18]:
'''LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0.
 An updated version of the class exists in the `langchain-chroma package and should be used instead.
  To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma'''




'LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0.\n An updated version of the class exists in the `langchain-chroma package and should be used instead.\n  To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma'

In [19]:
query = "What is the difference between positional and symbolic attention heads?"
results = parent_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Parent Chunk {i+1} ---")
    print(doc.page_content)
    print(f"\nLength: {len(doc.page_content)} characters")


--- Parent Chunk 1 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbolic head behavior
can be potentially relevant at the practical scale. Indeed, empirical evidence has shown that symbolic
mechanisms are likely to have better generalization properties than positional ones [Barbero et al.,
2024]. Moreover, positional and symbolic attention head behavior have been tightly linked to
mechanisms underlying state-of-the-art Rotary Positional Encoding (RoPE; Su et al. [2024]). In
particular, Urrutia et al. [2025] uncovered an interpretable causal relation between RoPE frequency
use, head behavior, and model performance of single layer models.
Whereas most research has focused on investigating the behavior of trained models, evaluating the

Length: 913 characters

--- Parent Chunk 2 ---
mechanisms Tra

In [20]:
## Basic QA  chain
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA


llm = ChatOpenAI(
    model='gpt-4o-mini',
    api_key=os.getenv('OPENAI_API_KEY')
)

qa_chain=RetrievalQA.from_chain_type(
            llm=llm,
            retriever=parent_retriever,
            return_source_documents=True
)

In [21]:
response = qa_chain.invoke({"query": query})

print("ANSWER:")
print(response['result'])
print("\nSOURCE CHUNKS USED:")
for i, doc in enumerate(response['source_documents']):
    print(f"\n--- Source {i+1} ---")
    print(doc.page_content[:300])

ANSWER:
Positional attention heads are tuned to behave based on the position of elements within the input sequence, focusing on the relative order and location of symbols. In contrast, symbolic attention heads are tuned to process the actual symbols themselves, emphasizing their content and relationships rather than their positions. This distinction is significant because empirical evidence suggests that symbolic mechanisms often exhibit better generalization properties than positional ones. Additionally, these behaviors are linked to the mechanisms underlying Rotary Positional Encoding (RoPE) in Transformers, with deeper layers typically exhibiting more symbolic behavior while early layers are more positional.

SOURCE CHUNKS USED:

--- Source 1 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbo

In [22]:
# Multi- Query Retriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [23]:
multi_query_retriever=MultiQueryRetriever.from_llm(
                        retriever=vectorstore.as_retriever(
                            search_kwargs={'k':3}
                        ),
                        llm=llm
)

In [ ]:
# Enable logging to see generated queries
import logging
logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

results = multi_query_retriever.invoke(query)

print(f"Total unique chunks retrieved: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. How do positional attention heads differ from symbolic attention heads in neural networks?  ', '2. Can you explain the distinctions between positional and symbolic attention mechanisms?  ', '3. What are the key differences in the functionalities of positional versus symbolic attention heads?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Total unique chunks retrieved: 4

--- Chunk 1 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbolic head behavior
can be potentially relevant at the practical scale. Indeed, empirical evidence has shown that symbolic
mechanisms are likely to have better generalization properties than positional ones [Barbero et al.,

--- Chunk 2 ---
how more complex tasks are solved through their combination.
2.2
Positional and symbolic attention head scores
To assess mechanistic differences between the tasks, we use the scores introduced by Urrutia et al.
[2025], which characterize whether an attention head behaves positionally or symbolically. Informally,
a head is positional if its attention distribution is invariant to input-token permutations, and symbolic

--- Chunk 3 ---
of attention head behavior. In par

In [25]:
'''Now you can see the real problem clearly.
Look at the three generated queries:

"What distinguishes positional attention heads from symbolic attention heads?"
"Can you explain how positional heads and symbolic heads differ in attention mechanisms?"
"What are the key differences between positional and symbolic heads in the context of attention models?"

These are three rephrasings of the exact same question. Definition, definition, definition. 
No mechanism angle. No consequence angle. The LLM defaulted to surface-level rephrasing.'''

'Now you can see the real problem clearly.\nLook at the three generated queries:\n\n"What distinguishes positional attention heads from symbolic attention heads?"\n"Can you explain how positional heads and symbolic heads differ in attention mechanisms?"\n"What are the key differences between positional and symbolic heads in the context of attention models?"\n\nThese are three rephrasings of the exact same question. Definition, definition, definition. \nNo mechanism angle. No consequence angle. The LLM defaulted to surface-level rephrasing.'

In [26]:
'''The result: All three queries hit the same semantic space in the vector store, retrieved the same chunks, deduplicated down to 3. 
Multi-query ran correctly — the retriever is not broken. The query generation is too narrow.'''

'The result: All three queries hit the same semantic space in the vector store, retrieved the same chunks, deduplicated down to 3. \nMulti-query ran correctly — the retriever is not broken. The query generation is too narrow.'

In [27]:
from langchain_core.prompts import PromptTemplate
custom_prompt=PromptTemplate(
    input_variables=["question"],
    template="""You are an AI assistant. Generate 3 different versions of the given question 
    to retrieve relevant documents. Each version must approach the question from a completely 
    different angle:
    - Version 1: Ask about the definition or characteristics
    - Version 2: Ask about the underlying mechanism or process
    - Version 3: Ask about the consequences or limitations
    
    Original question: {question}
    Provide exactly 3 questions, one per line, no numbering:"""
)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 3}
    ),
    llm=llm,
    prompt=custom_prompt
)

In [28]:
results = multi_query_retriever.invoke(query)

print(f"Total unique chunks retrieved: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What defines the characteristics and distinctions between positional and symbolic attention heads?  ', 'How do positional and symbolic attention heads function in the context of neural networks?  ', 'What are the limitations or implications of using positional versus symbolic attention heads in machine learning models?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Total unique chunks retrieved: 3

--- Chunk 1 ---
how more complex tasks are solved through their combination.
2.2
Positional and symbolic attention head scores
To assess mechanistic differences between the tasks, we use the scores introduced by Urrutia et al.
[2025], which characterize whether an attention head behaves positionally or symbolically. Informally,
a head is positional if its attention distribution is invariant to input-token permutations, and symbolic

--- Chunk 2 ---
these authors observed that in many real models, early layers heads are tuned to behave positionally,
while deeper layers heads are tuned to behave symbolically.
Understanding attention mechanisms through the lens of positional versus symbolic head behavior
can be potentially relevant at the practical scale. Indeed, empirical evidence has shown that symbolic
mechanisms are likely to have better generalization properties than positional ones [Barbero et al.,

--- Chunk 3 ---
of attention head behavior. In par